# Task 1 – GPO Annual Report: Baseline RAG
**Advanced RAG Hands-on Challenge**

| Step | What happens |
|------|--------------|
| 1 | Load GPO 2018 Annual Report PDF |
| 2 | Split into overlapping chunks with page metadata |
| 3 | Embed chunks → FAISS vector store (Gemini Embedding 2) |
| 4 | Build retriever (similarity search, k=4) |
| 5 | Wire LLM with strict document-only system prompt |
| 6 | Validate with 5 sample questions; inspect retrieved context |

> **Stack:** Python · LangChain (LCEL) · Google Gemini · FAISS

---
## 0 · Install dependencies

In [ ]:
%pip install -q \
    langchain \
    langchain-google-genai \
    langchain-community \
    langchain-text-splitters \
    faiss-cpu \
    pypdf \
    python-dotenv

---
## 0b · Configure API key

In [ ]:
import os, warnings, time
warnings.filterwarnings('ignore')

# Load from .env if present, otherwise use the key already in the environment
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# Fallback: set key directly if not in environment
if not os.getenv('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = 'AQ.Ab8RN6LZMzf8GK5UsxOgWaYetgIbmegtM13F8KVoUbNoCAiLOg'

print('API key ready.')

---
## Step 1 · Load the GPO Annual Report PDF

We use `PyPDFLoader` which extracts text page-by-page and preserves `page` metadata automatically.  
This lets us cite the source page in every retrieved chunk later.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = '/home/rahul-chokshi/Downloads/Rag Use cases-20261014T155740Z-1-001/Rag Use cases/GPO-Anual-report.pdf'

loader = PyPDFLoader(PDF_PATH)
pages  = loader.load()

total_chars = sum(len(p.page_content) for p in pages)
print(f'Loaded {len(pages)} pages — {total_chars:,} total characters')
print(f'\nMetadata sample (page 1): {pages[0].metadata}')
print(f'\n--- Page 1 preview (first 400 chars) ---')
print(pages[0].page_content[:400])

---
## Step 2 · Split into chunks

| Parameter | Value | Rationale |
|-----------|-------|----------|
| `chunk_size` | 800 chars | Fits within embedding model context; each chunk holds ~1 topic |
| `chunk_overlap` | 150 chars | Prevents answers that span a chunk boundary from being missed |
| `separators` | `\n\n → \n → . → ' '` | Tries to break at paragraph > sentence > word boundaries |

Source page number is carried forward in every chunk's metadata.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=['\n\n', '\n', '. ', ' ', ''],
)

chunks = splitter.split_documents(pages)

print(f'{len(chunks)} chunks created from {len(pages)} pages')
print(f'Average chunk length: {sum(len(c.page_content) for c in chunks)//len(chunks)} chars')
print()

# Show distribution of chunks per page
from collections import Counter
page_dist = Counter(c.metadata.get('page', '?') for c in chunks)
print('Chunks per page (first 10 pages):')
for pg, cnt in sorted(page_dist.items())[:10]:
    print(f'  Page {pg:>3}: {cnt} chunk(s)')

print(f'\n--- Sample chunk (chunk #5) ---')
sample = chunks[4]
print(f'Page: {sample.metadata.get("page")}')
print(f'Length: {len(sample.page_content)} chars')
print(sample.page_content)

---
## Step 3 · Embeddings → FAISS vector store

Each chunk is converted to a dense vector using **Gemini Embedding 2** and stored in a local FAISS index.  
FAISS is fully in-memory — no external server required.

> **Note:** We add a 3-second pause after bulk embedding to avoid transient 500 errors from the Gemini API.

In [ ]:
import re, time as _time
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-2')

# Free-tier limit: 100 embed requests/minute.
# We embed ONE chunk at a time with a 0.65s gap → ~92 req/min (safely under 100).
# On 429, we wait the retry delay from the error message, then continue.
PER_CHUNK_DELAY = 0.65

def _embed_one(text, emb, max_retries=6):
    """Embed a single text string with retry on rate-limit errors."""
    for attempt in range(max_retries):
        try:
            return emb.embed_documents([text])[0]
        except Exception as e:
            msg = str(e)
            if '429' in msg or 'RESOURCE_EXHAUSTED' in msg:
                m = re.search(r'retry in (\d+)', msg)
                wait = int(m.group(1)) + 5 if m else 65
                print(f'\n  [429] Rate limited — waiting {wait}s (attempt {attempt+1}/{max_retries})...', flush=True)
                _time.sleep(wait)
            else:
                raise
    raise RuntimeError(f'Embedding failed after {max_retries} retries.')

print(f'Embedding {len(chunks)} chunks one at a time ({PER_CHUNK_DELAY}s delay each)...')
print(f'Estimated time: ~{len(chunks) * PER_CHUNK_DELAY / 60:.1f} minutes\n')

t0     = _time.time()
texts   = [c.page_content for c in chunks]
metas   = [c.metadata     for c in chunks]
vectors = []

for i, text in enumerate(texts):
    vec = _embed_one(text, embeddings)
    vectors.append(vec)

    if (i + 1) % 20 == 0:
        elapsed   = _time.time() - t0
        remaining = (len(texts) - i - 1) * PER_CHUNK_DELAY
        print(f'  {i+1}/{len(texts)} embedded  ({elapsed:.0f}s elapsed, ~{remaining:.0f}s remaining)', flush=True)

    _time.sleep(PER_CHUNK_DELAY)

# Build FAISS index from pre-computed vectors (no extra API requests)
text_vec_pairs = list(zip(texts, vectors))
vector_store   = FAISS.from_embeddings(text_vec_pairs, embeddings, metadatas=metas)

elapsed = _time.time() - t0
print(f'\nDone in {elapsed:.1f}s  |  {vector_store.index.ntotal} vectors in FAISS index')
print('Waiting 3s before first query...')
_time.sleep(3)
print('Ready.')

---
## Step 4 · Retriever

For each query, the retriever embeds the question and returns the **k=4** most similar chunks by cosine similarity.  
We test it manually here before wiring it into the chain.

In [ ]:
retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 4},
)

def test_retrieval(query, k=4):
    docs = retriever.invoke(query)
    print(f'Query: "{query}"')
    print(f'Retrieved {len(docs)} chunks:')
    for i, doc in enumerate(docs, 1):
        page = doc.metadata.get('page', '?')
        print(f'  [{i}] Page {page}: {doc.page_content[:120].strip()}...')
    return docs

_ = test_retrieval('What were GPO revenues in FY 2018?')

---
## Steps 5 & 6 · LLM + document-only system prompt

**System prompt design:**
- The model sees **only** the retrieved chunks as its knowledge source.
- Explicit rule: answer from `<context>` only; if absent → `"I don't have that information."`
- `temperature=0` → deterministic, reduces hallucination risk.
- Chain follows LCEL (LangChain Expression Language) — no deprecated `langchain.chains` imports.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# ── LLM ──────────────────────────────────────────────────────────────────────
llm = ChatGoogleGenerativeAI(model='gemini-3.6-flash', temperature=0)

# ── System prompt (strict document-only grounding) ───────────────────────────
SYSTEM_PROMPT = """You are a precise document assistant for the GPO 2018 Annual Report.
Answer ONLY from the document excerpts in <context>.

Rules — no exceptions:
1. Base your answer exclusively on <context>. Do NOT use training knowledge.
2. If the answer is not present in <context>, respond with exactly:
   "I don't have that information."
3. Do not guess, extrapolate, or paraphrase beyond what is stated.
4. When possible, cite the relevant statistic or direct quote from the context.

<context>
{context}
</context>"""

qa_prompt = ChatPromptTemplate.from_messages([
    ('system', SYSTEM_PROMPT),
    MessagesPlaceholder('chat_history'),
    ('human', '{question}'),
])

# ── LCEL chain ───────────────────────────────────────────────────────────────
answer_chain = qa_prompt | llm | StrOutputParser()

def format_docs(docs):
    parts = []
    for d in docs:
        page = d.metadata.get('page', '?')
        parts.append(f'[Page {page}]\n{d.page_content}')
    return '\n\n---\n\n'.join(parts)

def call_with_retry(fn, payload, label='call', retries=3, delay=2):
    for attempt in range(retries):
        try:
            return fn(payload)
        except Exception as e:
            if attempt < retries - 1:
                wait = delay * (attempt + 1)
                print(f'[{label} retry {attempt+1}] {e} — waiting {wait}s...')
                time.sleep(wait)
            else:
                raise

def ask(question, chat_history=None):
    """Retrieve relevant chunks and generate a grounded answer."""
    if chat_history is None:
        chat_history = []
    docs    = call_with_retry(retriever.invoke, question, label='retrieval')
    context = format_docs(docs)
    payload = {'question': question, 'chat_history': chat_history, 'context': context}
    return call_with_retry(answer_chain.invoke, payload, label='LLM'), docs

print('Chain ready.')

---
## Validation · Required Assignment Questions + Extras

The first three questions are the **assignment pass/fail bar**.
The remaining two test additional coverage of the document.

| # | Question | Expected fact |
|---|----------|---------------|
| 1 | How many depository libraries did GPO staff visit in FY 2018? | Exact number from report |
| 2 | What new digital system replaced FDsys, and when was it fully deployed? | govinfo, fully deployed / FDsys retired Dec 2018 |
| 3 | How much did GPO procure in printing and information products on behalf of federal agencies? | Dollar amount from report |
| 4 | How many staff did GPO employ as of September 30, 2018? | 1,737 |
| 5 | How many passports did GPO produce in FY 2018? | 15,901,600 |

In [ ]:
SAMPLE_QUESTIONS = [
    # --- Assignment required questions (pass/fail bar) ---
    'How many depository libraries did GPO staff visit in FY 2018?',
    'What new digital system replaced FDsys, and when was it fully deployed?',
    'How much did GPO procure in printing and information products in FY 2018 on behalf of federal agencies?',
    # --- Additional coverage ---
    'How many staff did GPO employ as of September 30, 2018?',
    'How many passports did GPO produce in FY 2018?',
]

SEPARATOR = '=' * 70

results = []
for i, question in enumerate(SAMPLE_QUESTIONS, 1):
    print(f'{SEPARATOR}')
    label = '(REQUIRED)' if i <= 3 else '(EXTRA)'
    print(f'Q{i} {label}: {question}')
    print(SEPARATOR)

    answer, source_docs = ask(question)
    results.append({'question': question, 'answer': answer, 'sources': source_docs})

    print(f'Answer:\n  {answer}')
    print()
    print(f'Source chunks:')
    for j, doc in enumerate(source_docs, 1):
        page = doc.metadata.get('page', '?')
        print(f'  [{j}] Page {page}: {doc.page_content[:150].strip()}...')
    print()

    if i < len(SAMPLE_QUESTIONS):
        time.sleep(1)

---
## Out-of-scope test · Hallucination guard

The bot must refuse to answer anything not in the document.

In [ ]:
out_of_scope = [
    'What is the current GPO Director\'s salary in 2026?',
    'What is the stock ticker for GPO?',
]

print('--- Out-of-scope / hallucination guard test ---')
for q in out_of_scope:
    answer, _ = ask(q)
    print(f'Q: {q}')
    print(f'A: {answer}')
    print()

---
## Multi-turn chat · Conversational memory test

Verify that follow-up questions ("and what about...") resolve correctly with chat history.

In [ ]:
chat_history = []

conversation = [
    'What was GPO\'s net income in FY 2018?',
    'And what was the total revenue that year?',
    'How does that compare to the Congressional Publishing Appropriation?',
]

print('--- Multi-turn conversation ---')
for turn, q in enumerate(conversation, 1):
    print(f'Turn {turn} — You: {q}')
    answer, _ = ask(q, chat_history)
    print(f'Turn {turn} — Bot: {answer}')
    print()

    chat_history.append(HumanMessage(content=q))
    chat_history.append(AIMessage(content=answer))
    time.sleep(1)

---
## Summary

| Component | Choice | Notes |
|-----------|--------|-------|
| **Document loader** | `PyPDFLoader` | Page-level metadata preserved |
| **Chunker** | `RecursiveCharacterTextSplitter` | 800 chars / 150 overlap |
| **Embeddings** | `models/gemini-embedding-2` | Dense vectors via Google AI Studio |
| **Vector store** | FAISS (in-memory) | No external DB required |
| **Retriever** | Similarity search, k=4 | Returns 4 most relevant chunks |
| **LLM** | `models/gemini-2.5-flash` | temperature=0 for determinism |
| **Chain** | LCEL (`|` operator) | No deprecated `langchain.chains` |
| **Grounding** | Strict system prompt | Fallback: *"I don't have that information."* |

**Next:** Task 2 will extend this baseline with hybrid retrieval (BM25 + dense + RRF),
query rewriting, and re-ranking to improve retrieval quality.